# Setup

In [ ]:
%pip install astropy
%pip install git+https://github.com/pvigier/perlin-numpy
%pip install tensorflow_addons
%pip install wandb

In [ ]:
!git clone https://github.com/tomasmuzas/phd

In [ ]:
from phd.Augmentations.common_code.augmentation_functions import *
from phd.Augmentations.common_code.get_dataset import *
from phd.Augmentations.common_code.models import *
from phd.Augmentations.common_code.training_loop import *
from phd.Augmentations.common_code.utils import *
import tensorflow as tf

tf.get_logger().setLevel('ERROR')

In [ ]:
from google.colab import auth

auth.authenticate_user()

In [ ]:
!apt-get update -y
!curl -LJO "https://github.com/GoogleCloudPlatform/gcsfuse/releases/download/v1.2.0/gcsfuse_1.2.0_amd64.deb"
!apt-get -y install fuse
!apt-get clean
!dpkg -i "gcsfuse_1.2.0_amd64.deb"
!mkdir gcs
!gcsfuse --implicit-dirs <BUCKET NAME> gcs

In [ ]:
import wandb

wandb.login()

# Collect predictions

In [ ]:
tf.get_logger().setLevel('ERROR')

from tqdm.notebook import tqdm
import numpy as np
AUTO = tf.data.AUTOTUNE

for augmentations in [
    False,
    True]:
  for folds, train_size, test_size in [
      ("00-100", 4096, 715795),
      ("01-99", 8192, 711942),
      ("02-98", 15360, 704750),
      ("05-95", 36864, 683176),
      ("10-90", 72704, 647220),
      ("22-78", 158720, 560924),
      ("46-54", 331776, 388332)
    ]:
    print(f"augmentations: {augmentations}, setup: {folds}")
    training_config = {
    "ENABLE_DETERMINISM": False,
    "SEED": 17,
    "AUTO": tf.data.AUTOTUNE,
    "TRAIN_BATCH_SIZE": 1024,
    "TEST_BATCH_SIZE": 1024,
    "NUMBER_OF_EPOCHS": 500,
    "EARLY_STOPPING_TOLERANCE": 10,
    "LEARNING_RATE": 1e-4,
    "IMAGE_SIZE": 128,
    "SHUFFLE_BUFFER": train_size,
    "TPU": True,
    "AUGMENTATIONS_ROTATE": augmentations,
    "AUGMENTATIONS_FLIP_HORIZONTALLY": augmentations,
    "AUGMENTATIONS_FLIP_VERTICALLY": augmentations,
    "AUGMENTATIONS_ZOOM": augmentations,
    "AUGMENTATIONS_RANDOM_NOISE": augmentations,
    "AUGMENTATIONS_CENTER_NOISE": False,
    "AUGMENTATIONS_OUTSIDE_NOISE": False,
    "AUGMENTATIONS_PERLIN_CENTER_NOISE": False,
    "USE_ADABELIEF_OPTIMIZER": False,
    "GALAXY_DATASET": "SDSS",
    "TRAIN_DATASET_SIZE": train_size,
    "TEST_DATASET_SIZE": test_size,
    "STRETCH": False,
    "STEPS_PER_EXECUTION": 10,
    "FOLDS": 5,
    "NUMBER_OF_CLASSES": 5
    }
    training_config["WANDB_PROJECT_NAME"] = f"First paper, {training_config['NUMBER_OF_CLASSES']} classes"
    training_config["EXPERIMENT_DESCRIPTION"] = f"paper1_{'no_' if not augmentations else ''}augmentations_{training_config['NUMBER_OF_CLASSES']}_classes_{folds}"
    training_config["LOCAL_GCP_PATH_BASE"] = f"gcs/SpiralCertaintyBin"
    training_config["REMOTE_GCP_PATH_BASE"] = f"gs://<BUCKET NAME>/SpiralCertaintyBin"
    training_config["DATASET_PATH"] = f"data/TFDataset/MCCV/{training_config['NUMBER_OF_CLASSES']}_Classes/{folds}"

    models = [
        {'name': 'effnet', 'func': create_EffNetV2S, 'starting_fold': 1},
        {'name': 'cavanagh', 'func': Cavanagh, 'starting_fold': 1},
        {'name': 'dieleman', 'func': Dieleman, 'starting_fold': 1},
        {'name': 'resnet50', 'func': create_ResNet50, 'starting_fold': 1},
    ]

    for fold in range(1, training_config["FOLDS"] + 1):
      strategy = reset_tpu(training_config)

      with strategy.scope():

        test_dataset = get_dataset_with_objids(f"{training_config['REMOTE_GCP_PATH_BASE']}/{training_config['DATASET_PATH']}/fold_{fold}/test", 1024).prefetch(AUTO)
        if (training_config["TPU"]):
          test_dataset = test_dataset.cache()

        # print("Warming up dataset:")

        # for _ in tqdm(test_dataset, total = test_size // 1024 + 1):
        #   pass

        for model in models:
            experiment_path = f"models/128x128/experiments/{training_config['EXPERIMENT_DESCRIPTION']}"
            model_name = model['name']
            model_path = f"{experiment_path}/{model_name}"
            model_factory = model['func']

            if not os.path.exists(f"{training_config['LOCAL_GCP_PATH_BASE']}/{model_path}/best_loss/fold_{fold}/finished.txt"):
                print(f"FOLD {fold} FOR {model} IS NOT COMPLETED!")
                continue

            if os.path.exists(f"{training_config['LOCAL_GCP_PATH_BASE']}/{model_path}/best_loss/fold_{fold}/predictions.csv"):
                print(f"Predictions for fold {fold} model {model_name} alreay exist")
                continue

            if(os.path.exists(f"{training_config['LOCAL_GCP_PATH_BASE']}/{model_path}/best_loss/fold_{fold}/weights.index")):
                model = model_factory(training_config)
                model.load_weights(f"{training_config['REMOTE_GCP_PATH_BASE']}/{model_path}/best_loss/fold_{fold}/weights")

                galaxy_ids = np.empty([0, 1], dtype=str)
                true_labels = np.empty([0, 1], dtype=float)
                predictions = np.empty([0, 1], dtype=float)
                for images, labels, objids in tqdm(test_dataset, total = test_size // 1024 + 1):
                    results = model.predict(images, batch_size = 1024, steps = 1, verbose = 0)
                    galaxy_ids = tf.concat([tf.reshape(objids, [-1, 1]), galaxy_ids], axis=0)
                    true_labels = tf.concat([tf.reshape(labels, [-1, 1]), true_labels], axis=0)
                    predictions = tf.concat([tf.reshape(tf.argmax(results, axis= 1), [-1, 1]), predictions], axis=0)


                prediction_dataframe = pd.DataFrame({
                    "Id": galaxy_ids.numpy().astype(str).reshape((-1)),
                    "True": true_labels.numpy().reshape((-1)),
                    "Prediction": predictions.numpy().reshape((-1))
                })

                prediction_dataframe.to_csv(f"{training_config['LOCAL_GCP_PATH_BASE']}/{model_path}/best_loss/fold_{fold}/predictions.csv")

                del model
                del galaxy_ids
            else:
                print(f"WEIGHTS NOT FOUND")
      tf.keras.backend.clear_session()